# <Model name> — <your name>

Copy this notebook, rename it `notebooks/<model>_<yourname>.ipynb`, and replace
the two cells marked **YOUR MODEL**. Everything else is the shared protocol and
should not be edited — that is what makes your macro F1 comparable to everyone
else's.

The rules that still matter:

- The test split is touched exactly once, by `save_run`, after the best epoch is
  already fixed by **validation** macro F1. Never tune against test.
- Set `pretrained=` honestly in `save_run`. It is the one field that corrupts the
  results table if it is wrong.
- Don't edit `protocol.yaml` for your own run. If the team changes it, every model
  is re-run.

## 1. Setup

Works locally and on Colab. On Colab this clones the repo and expects the IP102
images on your Drive; adjust `DRIVE_IMAGES` to wherever you put them.

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = 'https://github.com/<org>/FarmPestManagementAI.git'
    DRIVE_IMAGES = '/content/drive/MyDrive/IP102/JPEGImages'  # <- where your images live

    from google.colab import drive; drive.mount('/content/drive')
    if not Path('FarmPestManagementAI').exists():
        subprocess.run(['git', 'clone', REPO], check=True)
    %cd FarmPestManagementAI
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

    # Point the protocol at the Drive copy instead of a local extraction.
    import yaml
    p = yaml.safe_load(open('protocol.yaml'))
    p['dataset']['subsets'][p['dataset']['subset']]['images'] = DRIVE_IMAGES
    yaml.safe_dump(p, open('protocol.yaml', 'w'), sort_keys=False)
else:
    # Running from notebooks/ — put the repo root on the path.
    sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import torch
import torch.nn as nn

from ip102_bench import (
    load_protocol, build_dataloaders, train_model, save_run,
    count_parameters, resolve_device, seed_everything,
)

protocol = load_protocol()
device   = resolve_device(protocol.runtime.get('device', 'auto'))
seed_everything(protocol.seed)

print(f"subset      : {protocol.subset_name} ({protocol.num_classes} classes)")
print(f"crop        : {protocol.crop_mode}, margin {protocol.crop_margin}")
print(f"image size  : {protocol.image_size}")
print(f"batch/epochs: {protocol.batch_size} / {protocol.epochs}")
print(f"seed        : {protocol.seed}")
print(f"device      : {device}")
print(f"classes     : {protocol.class_names}")

If that cell failed on a missing manifest, run this once from the repo root:

```bash
python scripts/setup_data.py
python scripts/check_data.py
```

In [ ]:
loaders = build_dataloaders(protocol)

for split, loader in loaders.items():
    print(f"{split:<11} {len(loader.dataset):>6} images, {len(loader):>4} batches")

print('\nclass counts:', loaders['train'].dataset.class_counts())

## 2. YOUR MODEL

Define your architecture here. The only hard requirements:

- accepts `[B, 3, image_size, image_size]`
- returns `[B, num_classes]` **raw logits** — no softmax, no log_softmax

For the pretrained baseline instead, delete this cell's body and use:

```python
from ip102_bench.models import build_pretrained
model = build_pretrained('resnet18', num_classes=protocol.num_classes)
```

...and remember to set `pretrained=True` further down.

In [ ]:
class MyNet(nn.Module):
    """<one line: what makes this architecture different from the others>"""

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)          # logits


MODEL_NAME = 'mynet'      # directory name and comparison-table row label
AUTHOR     = '<your name>'
PRETRAINED = False        # True if ANY weight started from someone else's training

model = MyNet(num_classes=protocol.num_classes)

### Sanity check before you train

Catches the two mistakes that waste an afternoon: a shape mismatch, and a head
that outputs the wrong number of classes.

In [ ]:
model = model.to(device)
dummy = torch.randn(2, 3, protocol.image_size, protocol.image_size, device=device)
out   = model(dummy)

assert out.shape == (2, protocol.num_classes), f'expected [2, {protocol.num_classes}], got {list(out.shape)}'
total, trainable = count_parameters(model)
print(f'output shape : {list(out.shape)}  ok')
print(f'parameters   : {total:,} total, {trainable:,} trainable')

### Does it overfit 64 images?

A correct model reaches ~100% training accuracy on a tiny subset within a couple
of hundred steps. If it cannot, the architecture or the wiring is broken and a
full run will only waste hours proving it. Optional but strongly recommended.

In [ ]:
from torch.utils.data import DataLoader, Subset

tiny   = DataLoader(Subset(loaders['train'].dataset, range(64)), batch_size=16, shuffle=True)
probe  = MyNet(num_classes=protocol.num_classes).to(device)
opt    = torch.optim.AdamW(probe.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

probe.train()
for epoch in range(60):
    correct = seen = 0
    for images, targets in tiny:
        images, targets = images.to(device), targets.to(device)
        opt.zero_grad(set_to_none=True)
        logits = probe(images)
        lossfn(logits, targets).backward()
        opt.step()
        correct += (logits.argmax(1) == targets).sum().item(); seen += targets.size(0)
    if epoch % 15 == 14:
        print(f'epoch {epoch+1:3d}  train acc on 64 images: {correct/seen:.3f}')

print('\nShould be close to 1.000. If it is stuck near 0.1, fix the model before training.')
del probe, opt

## 3. Train

`train_model` applies the locked protocol: AdamW, weighted cross-entropy,
ReduceLROnPlateau on validation macro F1, early stopping. It keeps the weights
from the best **validation** epoch.

Write your own loop instead if you want — but then match `protocol.yaml` exactly,
or your numbers do not belong in the same table.

In [ ]:
result = train_model(model, protocol, loaders, device=device)

print(f"\nbest epoch        : {result.best_epoch}")
print(f"best val macro F1 : {result.best_val_metric:.4f}")
print(f"epochs trained    : {result.epochs_trained} (early stop: {result.stopped_early})")
print(f"training time     : {result.training_seconds/60:.1f} min")

## 4. Test and save

This is the only place the test split is touched. It restores the best-validation
checkpoint first, then writes the full artifact set to `runs/<model>/<run_id>/` —
`results.json`, `training_history.csv`, `best_model.pt`, `predictions.csv` and the
three standard figures.

In [ ]:
run_dir = save_run(
    model=model,
    model_name=MODEL_NAME,
    protocol=protocol,
    result=result,
    test_loader=loaders['test'],
    pretrained=PRETRAINED,
    author=AUTHOR,
    architecture='<one line for the report: depth, widths, key idea>',
    notes='<anything unusual about this run>',
    device=device,
)

## 5. Error analysis

The report needs at least ten inspected failures. `predictions.csv` has every
test image with its prediction and confidence, so start with the confident
mistakes — those are where the model has learned something wrong, as opposed to
merely being unsure.

In [ ]:
import pandas as pd

preds  = pd.read_csv(run_dir / 'predictions.csv')
wrong  = preds[preds.correct == 0].sort_values('confidence', ascending=False)

print(f'{len(wrong)} errors out of {len(preds)} test images\n')
print('most confident mistakes:')
display(wrong.head(10)[['image_path', 'true_class', 'predicted_class', 'confidence']])

print('\nmost frequent confusions:')
display(wrong.groupby(['true_class', 'predicted_class']).size()
             .sort_values(ascending=False).head(10).rename('count').reset_index())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for ax, (_, row) in zip(axes.ravel(), wrong.head(10).iterrows()):
    ax.imshow(Image.open(protocol.image_root / row.image_path).convert('RGB'))
    ax.set_title(f"true: {row.true_class}\npred: {row.predicted_class} ({row.confidence:.2f})",
                 fontsize=8)
    ax.axis('off')
fig.suptitle('Most confident mistakes', fontweight='bold')
fig.tight_layout(); plt.show()

## 6. Where you stand

Compares your run against everyone else's. Run it again after your teammates push.

In [ ]:
!cd .. && python -m ip102_bench.compare